# 🏛️ LexiMini AI — Indian Legal Assistant
**Model:** google/gemma-4-E4B-it  
**Method:** QLoRA 4-bit + LoRA  
**Platform:** Google Colab (GPU)  
**Data:** Indian Laws 2026 (GCS se)

> Fine-tuning Gemma 4 on Indian Legal Data (IPC, BNS, Family Law, Labour Law etc.)

## CELL 1 — GPU Check

In [ ]:
import torch
print(f'CUDA available : {torch.cuda.is_available()}')
print(f'GPU name       : {torch.cuda.get_device_name(0)}')
print(f'VRAM           : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## CELL 2 — Install Libraries

In [ ]:
!pip install -q torch torchvision
!pip install -q 'transformers>=4.56.2'
!pip install -q datasets accelerate evaluate bitsandbytes trl peft
!pip install -q protobuf sentencepiece google-cloud-storage

import transformers, trl, peft, accelerate, bitsandbytes, datasets
print(f'transformers : {transformers.__version__}')
print(f'trl          : {trl.__version__}')
print(f'peft         : {peft.__version__}')
print(f'bitsandbytes : {bitsandbytes.__version__}')

## CELL 3 — HuggingFace Login + Config

In [ ]:
from google.colab import userdata
from huggingface_hub import login

# Colab Secrets mein HF_TOKEN add karo
login(token=userdata.get('HF_TOKEN'))

MODEL_ID    = 'google/gemma-4-E2B-it'
BUCKET_NAME = 'leximini-training-data'
PROJECT_ID  = 'linear-quasar-249806'

print(f'Model : {MODEL_ID}')
print(f'Bucket: gs://{BUCKET_NAME}')

## CELL 4 — GCS se Data Download

In [ ]:
from google.colab import auth
auth.authenticate_user()

!gsutil cp gs://{BUCKET_NAME}/data/train.jsonl /content/train.jsonl
!gsutil cp gs://{BUCKET_NAME}/data/eval.jsonl  /content/eval.jsonl

import json
def load_jsonl(path):
    with open(path) as f:
        return [json.loads(l) for l in f]

train_data = load_jsonl('/content/train.jsonl')
eval_data  = load_jsonl('/content/eval.jsonl')
print(f'Train: {len(train_data)} | Eval: {len(eval_data)}')
print('Sample:', train_data[0]['text'][:200])

## CELL 5 — Dataset Banana

In [ ]:
from datasets import Dataset

train_ds = Dataset.from_list(train_data)
eval_ds  = Dataset.from_list(eval_data)

print(f'Train dataset: {len(train_ds)} samples')
print(f'Eval dataset : {len(eval_ds)} samples')
print('\nSample entry:')
print(train_ds[0]['text'][:300])

In [ ]:
!pip install git+https://github.com/huggingface/transformers.git

## CELL 6 — Model + Tokenizer Load (QLoRA 4-bit)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit              = True,
    bnb_4bit_quant_type       = 'nf4',
    bnb_4bit_compute_dtype    = torch.bfloat16,
    bnb_4bit_use_double_quant = True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = 'right'

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config = bnb_config,
    device_map          = 'auto',
    torch_dtype         = torch.bfloat16,
    attn_implementation = 'eager',
)

print('✅ Model loaded')
print(f'Total parameters: {model.num_parameters():,}')

## CELL 7 — LoRA Config

In [ ]:
import bitsandbytes as bnb
from peft import LoraConfig, get_peft_model, TaskType

import torch # Added this import
!pip install --upgrade torchao>=0.16.0 # Added this line to upgrade torchao

def get_linear_layer_names(model):
    supported_cls = (torch.nn.Linear, bnb.nn.Linear4bit, bnb.nn.Linear8bitLt)
    return [
        name for name, module in model.named_modules()
        if isinstance(module, supported_cls)
    ]

model.enable_input_require_grads()
model.gradient_checkpointing_enable(
    gradient_checkpointing_kwargs={'use_reentrant': False}
)

target_modules = get_linear_layer_names(model)
print(f'Found {len(target_modules)} target modules for LoRA')

lora_config = LoraConfig(
    r              = 16,
    lora_alpha     = 32,
    target_modules = target_modules,
    lora_dropout   = 0.05,
    bias           = 'none',
    task_type      = TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## CELL 8 — Training

In [ ]:
from trl import SFTTrainer, SFTConfig

total_steps  = (len(train_ds) // (2 * 4)) * 3
warmup_steps = int(total_steps * 0.05)

training_args = SFTConfig(
    output_dir                  = '/content/leximini-checkpoints',
    num_train_epochs            = 3,
    per_device_train_batch_size = 2,
    per_device_eval_batch_size  = 2,
    gradient_accumulation_steps = 4,
    warmup_steps                = warmup_steps,
    learning_rate               = 2e-4,
    lr_scheduler_type           = 'cosine',
    bf16                        = True,
    logging_steps               = 10,
    eval_strategy               = 'epoch',
    save_strategy               = 'epoch',
    save_total_limit            = 2,
    load_best_model_at_end      = True,
    optim                       = 'adamw_bnb_8bit',
    gradient_checkpointing      = True,
    dataset_text_field          = 'text',
    packing                     = False,
)

trainer = SFTTrainer(
    model            = model,
    args             = training_args,
    train_dataset    = train_ds,
    eval_dataset     = eval_ds,
    processing_class = tokenizer
)

print('🚀 Training started..')
trainer.train()
print('✅ Training complete!')

In [ ]:
!gsutil -m cp -r /content/leximini-checkpoints gs://leximini-training-data/checkpoints/
print('✅ Checkpoint saved!')

## CELL 9 — Model Save to GCS + Google Drive

In [ ]:
import shutil
from google.colab import drive

SAVE_PATH = '/content/leximini-final'
trainer.model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f'✅ Model saved locally: {SAVE_PATH}')

# GCS pe save
!gsutil -m cp -r {SAVE_PATH} gs://{BUCKET_NAME}/models/gemma4-legal-lora/
print(f'✅ Uploaded to GCS: gs://{BUCKET_NAME}/models/gemma4-legal-lora/')

# Google Drive pe bhi save
drive.mount('/content/drive')
DRIVE_PATH = '/content/drive/MyDrive/leximini-final'
shutil.copytree(SAVE_PATH, DRIVE_PATH, dirs_exist_ok=True)
shutil.make_archive('/content/drive/MyDrive/leximini_adapter', 'zip', SAVE_PATH)
print('✅ Saved to Google Drive!')

## CELL 10 — Test Karo

In [ ]:
model.eval()

SYSTEM_PROMPT = (
    'You are LexiMini — an AI legal assistant specialized in Indian law. '
    'Answer questions about IPC, BNS, family law, labour law, and other Indian legal matters. '
    'Be clear, accurate, and helpful. Answer in the same language as the question (Hindi or English).'
)

def ask_leximini(question, max_new_tokens=300):
    prompt = (
        f'<start_of_turn>user\n{SYSTEM_PROMPT}\n\n{question}<end_of_turn>\n'
        f'<start_of_turn>model\n'
    )
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens     = max_new_tokens,
            temperature        = 0.7,
            do_sample          = True,
            repetition_penalty = 1.1,
            pad_token_id       = tokenizer.eos_token_id,
        )
    reply = tokenizer.decode(out[0], skip_special_tokens=True)
    return reply.split('<start_of_turn>model')[-1].strip()

# English test
print('=== ENGLISH ===')
print(ask_leximini('What are my rights if I am arrested under BNS Section 173?'))

print('\n=== HINDI ===')
print(ask_leximini('Agar mujhe bina warrant ke giraftaar kiya jaye toh mujhe kya karna chahiye?'))